# Roadmaps MVP End-to-End

This notebook walks through the current Python MVP pipeline for certification-program planning:

1. validate raw extracted course JSON files
2. inspect the skill taxonomy and alias normalization layer
3. transform raw courses into canonical course objects and quality snapshots
4. load the semester-based Data Science track configuration
5. match courses into the chosen track
6. inspect semester-by-semester candidate ranking, blocked reasons, and final roadmap
7. compare the successful baseline against an infeasible weaker baseline
8. run the public CLI commands and inspect their outputs

The notebook is designed to run from any working directory inside the repository tree.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path
from pprint import pprint


def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src' / 'roadmaps_mvp').exists():
            return candidate
    raise RuntimeError('Could not detect the roadmaps repo root.')


REPO_ROOT = find_repo_root(Path.cwd())
SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from roadmaps_mvp.io import iter_raw_course_paths, load_json
from roadmaps_mvp.models import ProgramSkillLevel
from roadmaps_mvp.normalize import SkillResolver, load_taxonomy
from roadmaps_mvp.planner import build_catalog, build_roadmap, load_program, rank_track_courses
from roadmaps_mvp.validate import validate_raw_course_dir

PROGRAM_PATH = REPO_ROOT / 'programs' / 'data_science_mvp.json'
TAXONOMY_PATH = REPO_ROOT / 'taxonomies' / 'skills.json'
REPORT_PATH = REPO_ROOT / 'reports' / 'roadmap_result.json'
NOTEBOOK_PATH = REPO_ROOT / 'output' / 'jupyter-notebook' / 'roadmaps-mvp-end-to-end.ipynb'


def show(title: str, payload, limit: int | None = None):
    print(f'\n{title}')
    print('-' * len(title))
    if hasattr(payload, 'model_dump'):
        payload = payload.model_dump(mode='json')
    if isinstance(payload, list) and limit is not None:
        payload = payload[:limit]
    print(json.dumps(payload, indent=2, ensure_ascii=False))


print('Current working directory:', Path.cwd())
print('Detected repo root:', REPO_ROOT)
print('Program path:', PROGRAM_PATH)
print('Notebook path:', NOTEBOOK_PATH)

Current working directory: /Users/pelmeshek1706/Desktop/projects/roadmaps/output/jupyter-notebook
Detected repo root: /Users/pelmeshek1706/Desktop/projects/roadmaps
Program path: /Users/pelmeshek1706/Desktop/projects/roadmaps/programs/data_science_mvp.json
Notebook path: /Users/pelmeshek1706/Desktop/projects/roadmaps/output/jupyter-notebook/roadmaps-mvp-end-to-end.ipynb


## 1. Validate raw course JSON files

The repository keeps the original discipline descriptions as a stable raw layer. We validate those files first and do not mutate them into program objects.

In [2]:
raw_paths = list(iter_raw_course_paths(REPO_ROOT))
raw_courses, validation_issues = validate_raw_course_dir(REPO_ROOT)

show(
    'Raw course files',
    [
        {
            'file': path.name,
            'course_id': path.stem,
        }
        for path in raw_paths
    ],
)
show(
    'Validation issues',
    [issue.model_dump(mode='json') for issue in validation_issues],
)
print(f'Validated courses: {len(raw_courses)}')


Raw course files
----------------
[
  {
    "file": "analysis_and_processing_of_time_series.json",
    "course_id": "analysis_and_processing_of_time_series"
  },
  {
    "file": "computer_vision_technologies.json",
    "course_id": "computer_vision_technologies"
  },
  {
    "file": "data_science_technologies.json",
    "course_id": "data_science_technologies"
  },
  {
    "file": "fundamentals_of_data_science.json",
    "course_id": "fundamentals_of_data_science"
  },
  {
    "file": "natural_language_analysis_and_processing_nlp.json",
    "course_id": "natural_language_analysis_and_processing_nlp"
  }
]

Validation issues
-----------------
[]
Validated courses: 5


## 2. Inspect one raw course object

This shows the extraction-layer shape before canonical normalization.

In [3]:
sample_raw_path = next(path for path in raw_paths if path.stem == 'data_science_technologies')
sample_raw = load_json(sample_raw_path)
show(
    'Raw course preview',
    {
        'course_name': sample_raw['course_name'],
        'discipline_tags': sample_raw['discipline_tags'],
        'input_skills_normalized_count': len(sample_raw['input_skills_normalized']),
        'output_skills_normalized_count': len(sample_raw['output_skills_normalized']),
        'curricular_relations_count': len(sample_raw['curricular_relations']),
        'confidence': sample_raw['confidence'],
    },
)
show('First 2 raw input skills', sample_raw['input_skills_normalized'], limit=2)
show('First 2 raw output skills', sample_raw['output_skills_normalized'], limit=2)


Raw course preview
------------------
{
  "course_name": "Data Science Technologies",
  "discipline_tags": [
    "data science",
    "statistical learning",
    "machine learning",
    "artificial intelligence",
    "decision support systems",
    "data mining",
    "time series analytics",
    "geospatial analytics"
  ],
  "input_skills_normalized_count": 7,
  "output_skills_normalized_count": 18,
  "curricular_relations_count": 11,
  "confidence": "medium"
}

First 2 raw input skills
------------------------
[
  {
    "skill_id": "python_basics",
    "skill_label": "Python basics",
    "min_level_required": 3,
    "importance": "required",
    "source": "explicit",
    "confidence": "high",
    "raw_mentions": [
      "Python syntax",
      "types and data structures",
      "branching operators",
      "functional programming",
      "OOP programming",
      "working with IDE",
      "environment creation"
    ],
    "evidence": [
      "The syllabus explicitly requires basic Pytho

## 3. Skill taxonomy and alias normalization

The taxonomy layer is the source of truth for canonical skill IDs. Raw aliases are resolved into canonical IDs before any cross-course planning happens.

In [4]:
taxonomy = load_taxonomy(TAXONOMY_PATH)
resolver = SkillResolver(taxonomy)

show(
    'Taxonomy summary',
    {
        'taxonomy_id': taxonomy.taxonomy_id,
        'version': taxonomy.version,
        'skills_count': len(taxonomy.skills),
    },
)
show(
    'Alias resolution examples',
    [
        {'query': 'basic_programming', 'resolved_to': resolver.resolve('basic_programming').canonical_skill_id},
        {'query': 'programming_basics', 'resolved_to': resolver.resolve('programming_basics').canonical_skill_id},
        {'query': 'linear_algebra_basics', 'resolved_to': resolver.resolve('linear_algebra_basics').canonical_skill_id},
        {'query': 'data_structures_and_algorithms', 'resolved_to': resolver.resolve('data_structures_and_algorithms').canonical_skill_id},
    ],
)
show('First 8 taxonomy entries', [entry.model_dump(mode='json') for entry in taxonomy.skills], limit=8)


Taxonomy summary
----------------
{
  "taxonomy_id": "skills",
  "version": "v1",
  "skills_count": 68
}

Alias resolution examples
-------------------------
[
  {
    "query": "basic_programming",
    "resolved_to": "programming_fundamentals"
  },
  {
    "query": "programming_basics",
    "resolved_to": "programming_fundamentals"
  },
  {
    "query": "linear_algebra_basics",
    "resolved_to": "linear_algebra"
  },
  {
    "query": "data_structures_and_algorithms",
    "resolved_to": "algorithms_and_data_structures"
  }
]

First 8 taxonomy entries
------------------------
[
  {
    "skill_id": "3d_reconstruction",
    "canonical_label": "3D Scene Reconstruction",
    "aliases": [],
    "category": "applied",
    "level_scale": {
      "min": 0,
      "max": 4
    },
    "status": "active"
  },
  {
    "skill_id": "algorithmic_thinking",
    "canonical_label": "Algorithmic Thinking",
    "aliases": [],
    "category": "applied",
    "level_scale": {
      "min": 0,
      "max": 4
  

## 4. Build the canonical course layer and quality layer

This step keeps the raw layer intact and creates the normalized objects used by the planner.

In [5]:
canonical_courses, quality_snapshots, catalog_issues = build_catalog(REPO_ROOT, TAXONOMY_PATH)
canonical_course = canonical_courses['data_science_technologies']
quality_snapshot = quality_snapshots[canonical_course.quality_ref]

show('Catalog build issues', [issue.model_dump(mode='json') for issue in catalog_issues])
show(
    'Canonical course preview',
    {
        'course_id': canonical_course.course_id,
        'course_name': canonical_course.course_name,
        'quality_ref': canonical_course.quality_ref,
        'domain_scores': [item.model_dump(mode='json') for item in canonical_course.domain_scores],
        'input_skill_refs_count': len(canonical_course.input_skill_refs),
        'output_skill_refs_count': len(canonical_course.output_skill_refs),
        'intra_course_relations_count': len(canonical_course.intra_course_relations),
    },
)
show('First 3 canonical input skill refs', [item.model_dump(mode='json') for item in canonical_course.input_skill_refs], limit=3)
show('Quality snapshot', quality_snapshot)


Catalog build issues
--------------------
[]

Canonical course preview
------------------------
{
  "course_id": "data_science_technologies",
  "course_name": "Data Science Technologies",
  "quality_ref": "course:data_science_technologies",
  "domain_scores": [
    {
      "domain_id": "data_science",
      "domain_label": "Data Science",
      "score": 1.0
    },
    {
      "domain_id": "statistical_learning",
      "domain_label": "Statistical Learning",
      "score": 0.8
    },
    {
      "domain_id": "machine_learning",
      "domain_label": "Machine Learning",
      "score": 0.78
    },
    {
      "domain_id": "decision_support_systems",
      "domain_label": "Decision Support Systems",
      "score": 0.56
    },
    {
      "domain_id": "business_intelligence",
      "domain_label": "Business Intelligence / Data Intelligence",
      "score": 0.42
    },
    {
      "domain_id": "geospatial_analytics",
      "domain_label": "Geospatial Analytics",
      "score": 0.34
    }
  

## 5. Load the semester-based certification program

The program object now contains a track definition, student baseline, semester planning window, course offerings, and inter-course edges.

In [6]:
program = load_program(PROGRAM_PATH)
track = next(track for track in program.tracks if track.track_id == program.track_id)

show(
    'Program summary',
    {
        'program_id': program.program_id,
        'title': program.title,
        'roadmap_mode': program.roadmap_mode,
        'selected_track_id': program.track_id,
        'planning_window': program.planning_window.model_dump(mode='json'),
        'completion_rules': program.completion_rules.model_dump(mode='json'),
    },
)
show(
    'Track summary',
    {
        'track_id': track.track_id,
        'selector_domains': [item.model_dump(mode='json') for item in track.selector_domains],
        'selector_tags': track.selector_tags,
        'target_skill_ids': [item.skill_id for item in track.target_skill_profile],
        'roadmap_policy': track.roadmap_policy.model_dump(mode='json'),
        'manual_includes': track.manual_includes,
    },
)
show(
    'Student baseline',
    {
        'student_id': program.student_profile.student_id,
        'baseline_skill_bank': [item.model_dump(mode='json') for item in program.student_profile.baseline_skill_bank],
        'completed_course_ids': program.student_profile.completed_course_ids,
    },
)
print('Note: this sample MVP assumes Python basics level 3 in the baseline profile because the current 5-course Data Science catalog does not contain a separate Python foundation course.')


Program summary
---------------
{
  "program_id": "data_science_certification_mvp",
  "title": "Data Science Certification MVP",
  "roadmap_mode": "semester",
  "selected_track_id": "data_science",
  "planning_window": {
    "start_term": {
      "course": 1,
      "semester": 1
    },
    "horizon_semesters": 4,
    "max_courses_per_semester": 3
  },
  "completion_rules": {
    "required_slot_ids": [],
    "min_selected_courses": 4,
    "target_coverage_threshold": 0.9
  }
}

Track summary
-------------
{
  "track_id": "data_science",
  "selector_domains": [
    {
      "domain_id": "data_science",
      "weight": 1.0
    },
    {
      "domain_id": "machine_learning",
      "weight": 0.8
    },
    {
      "domain_id": "statistical_learning",
      "weight": 0.7
    },
    {
      "domain_id": "predictive_analytics",
      "weight": 0.5
    },
    {
      "domain_id": "natural_language_processing",
      "weight": 0.4
    },
    {
      "domain_id": "computer_vision",
      "weight"

## 6. Group courses into the chosen track

This is the first real planning step: from the whole catalog, determine which disciplines belong to the selected Data Science track and why.

In [7]:
track_matches = rank_track_courses(program, canonical_courses)
show('Track matches', [match.model_dump(mode='json') for match in track_matches])


Track matches
-------------
[
  {
    "track_id": "data_science",
    "course_id": "analysis_and_processing_of_time_series",
    "affinity_score": 0.3714,
    "role": "core",
    "matched_domain_ids": [
      "data_science",
      "machine_learning",
      "predictive_analytics"
    ],
    "matched_target_skill_ids": [
      "ann_time_series_forecasting",
      "ols_regression",
      "time_series_decomposition"
    ],
    "matched_tags": [
      "data_science",
      "machine_learning",
      "predictive_modeling",
      "statistical_learning"
    ],
    "reason_codes": [
      "domain_alignment",
      "manual_include",
      "tag_similarity",
      "target_skill_overlap"
    ]
  },
  {
    "track_id": "data_science",
    "course_id": "data_science_technologies",
    "affinity_score": 0.3647,
    "role": "foundation",
    "matched_domain_ids": [
      "data_science",
      "machine_learning",
      "statistical_learning"
    ],
    "matched_target_skill_ids": [
      "data_preproces

## 7. Build the semester roadmap

The planner now works semester by semester. It respects:

- max 4 semesters
- max 3 subjects per semester
- offering availability by term
- stage order: foundation -> core -> advanced
- prerequisite accumulation only after a semester is completed

In [8]:
roadmap = build_roadmap(program, canonical_courses, quality_snapshots)
show(
    'Roadmap headline',
    {
        'selected_track_id': roadmap.selected_track_id,
        'selected_course_ids': roadmap.selected_course_ids,
        'achieved_target_coverage': roadmap.achieved_target_coverage,
        'unmet_constraints': roadmap.unmet_constraints,
    },
)
for plan in roadmap.semester_plans:
    show(
        f'Semester {plan.semester_index} @ term {plan.term.course}.{plan.term.semester}',
        {
            'selected_course_ids': plan.selected_course_ids,
            'coverage_after': plan.coverage_after,
            'top_candidates': [
                {
                    'course_id': item.course_id,
                    'score': item.score,
                    'stage': item.stage,
                    'target_skill_gain': item.target_skill_gain,
                    'track_affinity': item.track_affinity,
                    'missing_prerequisites': item.missing_prerequisites,
                }
                for item in plan.candidate_courses
            ],
            'blocked_course_reasons': plan.blocked_course_reasons,
        },
    )


Roadmap headline
----------------
{
  "selected_track_id": "data_science",
  "selected_course_ids": [
    "fundamentals_of_data_science",
    "data_science_technologies",
    "natural_language_analysis_and_processing_nlp",
    "analysis_and_processing_of_time_series",
    "computer_vision_technologies"
  ],
  "achieved_target_coverage": 1.0,
  "unmet_constraints": []
}

Semester 1 @ term 1.1
---------------------
{
  "selected_course_ids": [
    "fundamentals_of_data_science"
  ],
  "coverage_after": 0.1648,
  "top_candidates": [
    {
      "course_id": "fundamentals_of_data_science",
      "score": 0.3547,
      "stage": "foundation",
      "target_skill_gain": 0.1648,
      "track_affinity": 0.18,
      "missing_prerequisites": []
    }
  ],
  "blocked_course_reasons": [
    "analysis_and_processing_of_time_series:not_offered",
    "computer_vision_technologies:not_offered",
    "data_science_technologies:not_offered",
    "natural_language_analysis_and_processing_nlp:not_offered"


## 8. Final student-facing summary

This is the compact artifact the planner returns after it has consumed all selected semesters.

In [9]:
show('Planner summary', roadmap.summary)
target_skill_ids = {item.skill_id for item in track.target_skill_profile}
show(
    'Final target skill bank',
    [item.model_dump(mode='json') for item in roadmap.final_skill_bank if item.skill_id in target_skill_ids],
)
print('Semester-by-semester final path:')
for plan in roadmap.semester_plans:
    print(f"  semester {plan.semester_index} ({plan.term.course}.{plan.term.semester}): {', '.join(plan.selected_course_ids) or 'no courses'}")


Planner summary
---------------
{
  "total_semesters": 4,
  "total_courses": 5,
  "achieved_target_coverage": 1.0,
  "gained_target_skills": [
    "ann_time_series_forecasting",
    "data_preprocessing",
    "data_visualization",
    "digital_image_processing",
    "feature_extraction",
    "image_recognition_and_detection",
    "lemmatization",
    "multicriteria_decision_analysis",
    "ols_regression",
    "similarity_computation",
    "text_vectorization",
    "time_series_decomposition"
  ],
  "unmet_target_skills": []
}

Final target skill bank
-----------------------
[
  {
    "skill_id": "ann_time_series_forecasting",
    "level": 3
  },
  {
    "skill_id": "data_preprocessing",
    "level": 3
  },
  {
    "skill_id": "data_visualization",
    "level": 3
  },
  {
    "skill_id": "digital_image_processing",
    "level": 3
  },
  {
    "skill_id": "feature_extraction",
    "level": 3
  },
  {
    "skill_id": "image_recognition_and_detection",
    "level": 3
  },
  {
    "skill_i

## 9. Infeasibility check: weaken the baseline

This shows the opposite case. If the student starts with weaker Python readiness and the current catalog still has no Python foundation course, the planner should not pretend the track is complete.

In [10]:
weaker_profile = program.student_profile.model_copy(deep=True)
weaker_profile.baseline_skill_bank = [
    item.model_copy(update={'level': 2}) if item.skill_id == 'python_basics' else item
    for item in weaker_profile.baseline_skill_bank
]
weaker_program = program.model_copy(update={'student_profile': weaker_profile}, deep=True)
weaker_roadmap = build_roadmap(weaker_program, canonical_courses, quality_snapshots)

show(
    'Weaker-baseline roadmap result',
    {
        'selected_course_ids': weaker_roadmap.selected_course_ids,
        'achieved_target_coverage': weaker_roadmap.achieved_target_coverage,
        'unmet_constraints': weaker_roadmap.unmet_constraints,
    },
)
for plan in weaker_roadmap.semester_plans:
    print(f"semester {plan.semester_index} blocked reasons:")
    pprint(plan.blocked_course_reasons)



Weaker-baseline roadmap result
------------------------------
{
  "selected_course_ids": [
    "fundamentals_of_data_science",
    "analysis_and_processing_of_time_series",
    "natural_language_analysis_and_processing_nlp",
    "computer_vision_technologies"
  ],
  "achieved_target_coverage": 0.7143,
  "unmet_constraints": [
    "external_prereq_gap:data_science_technologies:python_basics:3",
    "semester_2:no_eligible_courses",
    "target_coverage_below_policy"
  ]
}
semester 1 blocked reasons:
['analysis_and_processing_of_time_series:not_offered',
 'computer_vision_technologies:not_offered',
 'data_science_technologies:not_offered',
 'natural_language_analysis_and_processing_nlp:not_offered']
semester 2 blocked reasons:
['analysis_and_processing_of_time_series:not_offered',
 'computer_vision_technologies:not_offered',
 'data_science_technologies:missing_prerequisites',
 'natural_language_analysis_and_processing_nlp:not_offered']
semester 3 blocked reasons:
['computer_vision_techn

## 10. CLI walkthrough

The same pipeline is also exposed through the public CLI. This is useful when you want to validate the repo or rebuild the roadmap without importing Python modules manually.

In [11]:
from subprocess import run

cli_commands = [
    ['python3', '-m', 'roadmaps_mvp.cli', 'validate-courses'],
    ['python3', '-m', 'roadmaps_mvp.cli', 'rank-track-courses'],
    ['python3', '-m', 'roadmaps_mvp.cli', 'build-roadmap'],
]

for command in cli_commands:
    print(f"\n$ PYTHONPATH=src {' '.join(command)}")
    completed = run(
        command,
        cwd=str(REPO_ROOT),
        env={'PYTHONPATH': str(SRC_DIR), **__import__('os').environ},
        capture_output=True,
        text=True,
        check=True,
    )
    print(completed.stdout[:4000])


$ PYTHONPATH=src python3 -m roadmaps_mvp.cli validate-courses


{
  "validated_courses": 5,
  "error_count": 0,
  "warning_count": 0,
  "errors": [],
  "warnings": []
}


$ PYTHONPATH=src python3 -m roadmaps_mvp.cli rank-track-courses


[
  {
    "track_id": "data_science",
    "course_id": "analysis_and_processing_of_time_series",
    "affinity_score": 0.3714,
    "role": "core",
    "matched_domain_ids": [
      "data_science",
      "machine_learning",
      "predictive_analytics"
    ],
    "matched_target_skill_ids": [
      "ann_time_series_forecasting",
      "ols_regression",
      "time_series_decomposition"
    ],
    "matched_tags": [
      "data_science",
      "machine_learning",
      "predictive_modeling",
      "statistical_learning"
    ],
    "reason_codes": [
      "domain_alignment",
      "manual_include",
      "tag_similarity",
      "target_skill_overlap"
    ]
  },
  {
    "track_id": "data_science",
    "course_id": "data_science_technologies",
    "affinity_score": 0.3647,
    "role": "foundation",
    "matched_domain_ids": [
      "data_science",
      "machine_learning",
      "statistical_learning"
    ],
    "matched_target_skill_ids": [
      "data_preprocessing",
      "data_visualizat

{
  "output": "reports/roadmap_result.json",
  "program_id": "data_science_certification_mvp",
  "selected_track_id": "data_science",
  "selected_course_ids": [
    "fundamentals_of_data_science",
    "data_science_technologies",
    "natural_language_analysis_and_processing_nlp",
    "analysis_and_processing_of_time_series",
    "computer_vision_technologies"
  ],
  "slot_results": [],
  "semester_plans": [
    {
      "semester_index": 1,
      "term": {
        "course": 1,
        "semester": 1
      },
      "selected_course_ids": [
        "fundamentals_of_data_science"
      ],
      "selected_courses": [
        {
          "course_id": "fundamentals_of_data_science",
          "course_name": "Fundamentals of Data Science",
          "score": 0.3547,
          "track_affinity": 0.18,
          "prerequisite_fit": 1.0,
          "readiness_score": 1.0,
          "target_skill_gain": 0.1648,
          "unlock_score": 0.0,
          "domain_alignment": 0.18,
          "phase_fit_s

## 11. Generated files

The CLI writes the final roadmap report and keeps the raw, canonical, taxonomy, and quality layers separate.

In [12]:
show('Latest roadmap report', load_json(REPORT_PATH))
print('Useful files:')
print('  raw layer:            ', REPO_ROOT)
print('  taxonomy layer:       ', TAXONOMY_PATH)
print('  program layer:        ', PROGRAM_PATH)
print('  latest report:        ', REPORT_PATH)
print('  this notebook:        ', NOTEBOOK_PATH)


Latest roadmap report
---------------------
{
  "program_id": "data_science_certification_mvp",
  "selected_track_id": "data_science",
  "selected_course_ids": [
    "fundamentals_of_data_science",
    "data_science_technologies",
    "natural_language_analysis_and_processing_nlp",
    "analysis_and_processing_of_time_series",
    "computer_vision_technologies"
  ],
  "slot_results": [],
  "semester_plans": [
    {
      "semester_index": 1,
      "term": {
        "course": 1,
        "semester": 1
      },
      "selected_course_ids": [
        "fundamentals_of_data_science"
      ],
      "selected_courses": [
        {
          "course_id": "fundamentals_of_data_science",
          "course_name": "Fundamentals of Data Science",
          "score": 0.3547,
          "track_affinity": 0.18,
          "prerequisite_fit": 1.0,
          "readiness_score": 1.0,
          "target_skill_gain": 0.1648,
          "unlock_score": 0.0,
          "domain_alignment": 0.18,
          "phase_fit